# CG-Projection Enclosure with Projected Richardson Correction

This notebook applies the CG-projection enclosure with projected Richardson correction to the 300 free coordinates of `system_303`. It uses symmetric Jacobi scaling, records the search directions from the legacy SciPy CG recurrence, and evaluates the enclosure every 10 iterations.

For each CG prefix, the corrected center and enclosure map are

$$
\widehat x_m^{(j+1)}=\widehat x_m^{(j)}+\omega\Pi_m\left(b-A\widehat x_m^{(j)}\right),
\qquad
E_m^{(j+1)}=\Pi_m(I-\omega A)E_m^{(j)},
$$

starting from $E_m^{(0)}=\Pi_m$. The interval-hull radius is therefore

$$
d_m^{\mathrm{proj}}=\left|E_m^{(q)}\right|d^0.
$$


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from scipy.linalg import qr, solve
from scipy.sparse.linalg._isolve.utils import make_system


data_dir = Path("system_303")

A_full = np.genfromtxt(data_dir / "stiffness_beam_A.dat", delimiter=",")
b_full = np.genfromtxt(data_dir / "forcing_b.dat")
box = np.genfromtxt(data_dir / "B0_wide_bc.dat", delimiter=",")
x_star_full = np.genfromtxt(data_dir / "soln_x.dat")

# Remove the first three boundary-condition coordinates, as in v3.
A = A_full[3:, 3:]
b = b_full[3:]
ell0 = box[3:, 0]
u0 = box[3:, 1]
x_star = x_star_full[3:]

print(f"system dimension: {len(b)}")


system dimension: 300


## Jacobi scaling

With `scale = sqrt(diag(A))`, solve in the coordinates $y=\mathrm{scale}\,x$ and transform the final enclosure back to the original coordinates.


In [3]:
diagonal = np.diag(A)
if np.any(diagonal <= 0):
    raise ValueError("Jacobi scaling requires a positive diagonal.")

scale = np.sqrt(diagonal)
A_hat = A / scale[:, None] / scale[None, :]
b_hat = b / scale
ell_hat = scale * ell0
u_hat = scale * u0
c0 = 0.5 * (ell_hat + u_hat)

# Algorithm parameters. Here m=n, q=1, and omega is safely inside
# (0, 2 / lambda_max(A_hat)).
max_iterations = 520
contraction_period = 10
projected_richardson_steps = 1
lambda_max_hat = float(np.linalg.eigvalsh(A_hat)[-1])
omega = 1.0 / lambda_max_hat

# if max_iterations > len(b_hat):
#     raise ValueError("The algorithm requires m <= n.")
if projected_richardson_steps < 0:
    raise ValueError("The number of projected Richardson steps must be nonnegative.")
if not 0.0 < omega < 2.0 / lambda_max_hat:
    raise ValueError("omega must satisfy 0 < omega < 2 / lambda_max(A_hat).")


## CG-projection and projected Richardson correction

The first function retains the legacy unpreconditioned SciPy CG recurrence used by the earlier notebook while recording $p_k$ and $Ap_k$. The second selects a stable linearly independent subset of the recorded directions. The remaining functions evaluate $\widehat x_m^{(0)}$, $\Pi_m$, the $q$ projected Richardson corrections, and the interval hull $|E_m^{(q)}|d^0$ for each reported CG prefix.


In [6]:
def scipy_cg_with_directions(A, b, x0, maxiter):
    """Run SciPy's legacy unpreconditioned CG recurrence and retain p_k and A p_k.

    The recurrence is copied from scipy.sparse.linalg.cg. Its tolerance exit
    is intentionally omitted because the enclosure algorithm prescribes m
    iterations. Jacobi scaling has already been applied.
    """
    A, M, x, b, postprocess = make_system(A, None, x0, b)
    n = len(b)
    if maxiter < 1:
        raise ValueError("maxiter must be positive.")
    # if maxiter > n:
    #     raise ValueError("The enclosure algorithm requires maxiter <= n.")

    dotprod = np.vdot if np.iscomplexobj(x) else np.dot
    matvec = A.matvec
    psolve = M.matvec
    r = b - matvec(x) if x.any() else b.copy()

    directions = np.empty((n, maxiter), dtype=x.dtype)
    applied_directions = np.empty((n, maxiter), dtype=x.dtype)
    residual_norms = np.empty(maxiter, dtype=float)

    # SciPy's CG recurrence; the extra assignments only record enclosure data.
    rho_prev, p = None, None
    for iteration in range(maxiter):
        z = psolve(r)
        rho_cur = dotprod(r, z)
        if iteration > 0:
            beta = rho_cur / rho_prev
            p *= beta
            p += z
        else:
            p = np.empty_like(r)
            p[:] = z[:]

        directions[:, iteration] = p
        q_direction = matvec(p)
        applied_directions[:, iteration] = q_direction
        alpha = rho_cur / dotprod(p, q_direction)
        x += alpha * p
        r -= alpha * q_direction
        rho_prev = rho_cur
        residual_norms[iteration] = np.linalg.norm(r)

    return {
        "x_cg": postprocess(x),
        "residual": r,
        "residual_norms": residual_norms,
        "directions": directions,
        "applied_directions": applied_directions,
    }


def select_independent_directions(
    directions,
    applied_directions,
    qr_tolerance=1e-10,
    max_gram_condition=1e12,
):
    """A-normalize and retain a stable pivoted-QR subset."""
    a_norm_squared = np.sum(directions * applied_directions, axis=0)
    valid = np.isfinite(a_norm_squared) & (a_norm_squared > 0)
    valid_indices = np.flatnonzero(valid)
    if valid_indices.size == 0:
        raise RuntimeError("CG produced no direction with a positive A-norm.")

    a_norm = np.sqrt(a_norm_squared[valid])
    normalized = directions[:, valid] / a_norm[None, :]
    normalized_applied = applied_directions[:, valid] / a_norm[None, :]
    _, triangular, pivots = qr(normalized, mode="economic", pivoting=True)
    diagonal = np.abs(np.diag(triangular))
    candidate_rank = int(np.sum(diagonal > qr_tolerance * diagonal[0]))
    if candidate_rank == 0:
        raise RuntimeError("The recorded CG directions are numerically rank zero.")

    for rank in range(candidate_rank, 0, -1):
        local = pivots[:rank]
        basis = normalized[:, local]
        applied_basis = normalized_applied[:, local]
        gram = basis.T @ applied_basis
        gram = 0.5 * (gram + gram.T)
        condition = np.linalg.cond(gram)
        if np.isfinite(condition) and condition <= max_gram_condition:
            return basis, applied_basis, gram

    raise RuntimeError("No numerically stable independent direction set was found.")


def contract_projected_richardson(
    A,
    b,
    ell0,
    u0,
    directions,
    applied_directions,
    richardson_steps,
    omega,
):
    """Compute B_m using CG projection followed by q projected Richardson steps."""
    c0 = 0.5 * (ell0 + u0)
    d0 = 0.5 * (u0 - ell0)
    P, AP, G = select_independent_directions(directions, applied_directions)

    coefficients = solve(
        G,
        P.T @ (b - A @ c0),
        assume_a="sym",
        check_finite=False,
    )
    x_hat = c0 + P @ coefficients

    Ginv_PTA = solve(
        G,
        AP.T,
        assume_a="sym",
        check_finite=False,
    )
    Pi = np.eye(len(b)) - P @ Ginv_PTA
    E = Pi.copy()

    for _ in range(richardson_steps):
        x_hat = x_hat + omega * (Pi @ (b - A @ x_hat))
        E = Pi @ (E - omega * (A @ E))

    d_projected = np.abs(E) @ d0
    ell = np.maximum(ell0, x_hat - d_projected)
    u = np.minimum(u0, x_hat + d_projected)
    if np.any(ell > u):
        violation = ell - u
        raise RuntimeError(
            "The projected Richardson enclosure produced an inverted interval; "
            f"maximum violation={np.max(violation):.3e}."
        )

    return ell, u


def contract_every(
    A,
    b,
    ell0,
    u0,
    directions,
    applied_directions,
    period,
    richardson_steps,
    omega,
):
    """Evaluate B_m from the same initial box at each reported CG prefix."""
    final_iteration = directions.shape[1]
    iterations = list(range(0, final_iteration + 1, period))
    if iterations[-1] != final_iteration:
        iterations.append(final_iteration)

    ell_history = [np.array(ell0, copy=True)]
    u_history = [np.array(u0, copy=True)]
    for iteration in iterations[1:]:
        ell, u = contract_projected_richardson(
            A=A,
            b=b,
            ell0=ell0,
            u0=u0,
            directions=directions[:, :iteration],
            applied_directions=applied_directions[:, :iteration],
            richardson_steps=richardson_steps,
            omega=omega,
        )
        ell_history.append(ell)
        u_history.append(u)

    return np.asarray(iterations), np.asarray(ell_history), np.asarray(u_history)


## Run the enclosure


In [7]:
cg_run = scipy_cg_with_directions(
    A=A_hat,
    b=b_hat,
    x0=c0,
    maxiter=max_iterations,
)
x_cg = cg_run["x_cg"]
P_all = cg_run["directions"]
AP_all = cg_run["applied_directions"]

iteration_history, ell_history_y, u_history_y = contract_every(
    A=A_hat,
    b=b_hat,
    ell0=ell_hat,
    u0=u_hat,
    directions=P_all,
    applied_directions=AP_all,
    period=contraction_period,
    richardson_steps=projected_richardson_steps,
    omega=omega,
)

# Convert all stored enclosures back to the original x coordinates.
ell_history = ell_history_y / scale[None, :]
u_history = u_history_y / scale[None, :]
ell_final = ell_history[-1]
u_final = u_history[-1]

initial_width = u0 - ell0
width_history = u_history - ell_history
with np.errstate(divide="ignore", invalid="ignore"):
    shrinkage_history = 100.0 * (1.0 - width_history / initial_width[None, :])

contains_solution = np.all((ell_final <= x_star) & (x_star <= u_final))
print(f"CG iterations m: {max_iterations}")
print(f"projected Richardson steps q: {projected_richardson_steps}")
print(f"lambda_max(A_hat): {lambda_max_hat:.8e}")
print(f"omega: {omega:.8e}")
print(f"2 / lambda_max(A_hat): {2.0 / lambda_max_hat:.8e}")
print(f"contraction calls: {len(iteration_history) - 1}")
print(f"final SciPy-CG residual norm: {cg_run['residual_norms'][-1]:.8e}")
print(f"final mean shrinkage: {np.nanmean(shrinkage_history[-1]):.2f}%")
print(f"final enclosure contains x_star: {contains_solution}")


CG iterations m: 520
projected Richardson steps q: 1
lambda_max(A_hat): 2.68614066e+00
omega: 3.72281323e-01
2 / lambda_max(A_hat): 7.44562647e-01
contraction calls: 52
final SciPy-CG residual norm: 3.58057023e-01
final mean shrinkage: 66.73%
final enclosure contains x_star: False


## Controllable animations

Each animation includes play/pause buttons and a frame slider. Coordinate indices use Python's zero-based convention, so the `1 mod 3` coordinates are `1, 4, 7, ...`.


In [8]:
all_indices = np.arange(len(initial_width))
mod1_indices = all_indices[all_indices % 3 == 1]


def display_shrinkage_animation(indices, title):
    frame_data = shrinkage_history[:, indices]
    finite_data = frame_data[np.isfinite(frame_data)]
    y_max = max(1.0, float(np.max(finite_data)) * 1.05) if finite_data.size else 1.0

    fig, ax = plt.subplots(figsize=(9, 4.5))
    line, = ax.plot(indices, np.nan_to_num(frame_data[0]), linewidth=1)
    label = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(xlim=(indices[0], indices[-1]), ylim=(0, y_max))
    ax.set_xlabel("Coordinate index")
    ax.set_ylabel("Width reduction (% of original width)")
    ax.set_title(title)
    ax.grid(alpha=0.25)

    def update(frame):
        line.set_ydata(np.nan_to_num(frame_data[frame]))
        label.set_text(f"CG iteration {iteration_history[frame]}")
        return line, label

    animation = FuncAnimation(fig, update, frames=len(iteration_history), interval=180)
    display(HTML(animation.to_jshtml(fps=5, default_mode="loop")))
    plt.close(fig)
    return animation


def display_bounds_animation(indices, title):
    lower_data = ell_history[:, indices]
    upper_data = u_history[:, indices]
    y_min = float(np.min(lower_data))
    y_max = float(np.max(upper_data))
    padding = 0.03 * max(y_max - y_min, 1.0)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    lower_line, = ax.plot(indices, lower_data[0], label="lower bound", linewidth=1)
    upper_line, = ax.plot(indices, upper_data[0], label="upper bound", linewidth=1)
    label = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
    ax.set(xlim=(indices[0], indices[-1]), ylim=(y_min - padding, y_max + padding))
    ax.set_xlabel("Coordinate index")
    ax.set_ylabel("Bound value")
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend()

    def update(frame):
        lower_line.set_ydata(lower_data[frame])
        upper_line.set_ydata(upper_data[frame])
        label.set_text(f"CG iteration {iteration_history[frame]}")
        return lower_line, upper_line, label

    animation = FuncAnimation(fig, update, frames=len(iteration_history), interval=180)
    display(HTML(animation.to_jshtml(fps=5, default_mode="loop")))
    plt.close(fig)
    return animation


### Shrinkage along all coordinates


In [9]:
all_coordinate_animation = display_shrinkage_animation(
    all_indices,
    "Shrinkage along all coordinates",
)


### Shrinkage along coordinates 1 mod 3


In [10]:
mod1_shrinkage_animation = display_shrinkage_animation(
    mod1_indices,
    "Shrinkage along coordinates 1 mod 3",
)


### Upper and lower bounds for coordinates 1 mod 3


In [11]:
mod1_bounds_animation = display_bounds_animation(
    mod1_indices,
    "Upper and lower bounds for coordinates 1 mod 3",
)
